In [16]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier

ROOT = Path.cwd().parent
raw_path = ROOT / 'data' / 'raw' / 'raw_data.csv'
models_dir = ROOT / 'models'
preprocessor_path = models_dir / 'preprocessor.joblib'

df = pd.read_csv(raw_path).drop(columns=['Student_ID'])
X = df.drop(columns=['Mental_Health_Risk'])
y = df['Mental_Health_Risk'].map({'Low': 0, 'Medium': 1, 'High': 2})

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state= 42, stratify=y)

Load saved preprocessor and transform splits

In [17]:
preprocessor = joblib.load(preprocessor_path)
X_train_ready = preprocessor.transform(X_train)
X_test_ready = preprocessor.transform(X_test)

Initialize the 3 models

In [18]:
models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest': RandomForestClassifier(),
    'XGBoost': XGBClassifier(),
}

Train, predict, and compute all 5 metrics

In [19]:
results = []
for name, model in models.items():
  model.fit(X_train_ready, y_train)
  y_pred = model.predict(X_test_ready)
  y_proba = model.predict_proba(X_test_ready)

  results.append({
      'Model': name,
      'Accuracy': accuracy_score(y_test, y_pred),
      'Precision': precision_score(y_test, y_pred, average='macro'),
      'Recall': recall_score(y_test, y_pred, average='macro'),
      'F1-Score': f1_score(y_test, y_pred, average='macro')
  })

results_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
print(results_df)

                 Model  Accuracy  Precision    Recall  F1-Score
0  Logistic Regression    0.7700   0.771483  0.769236  0.770134
2              XGBoost    0.7075   0.705793  0.706625  0.706163
1        Random Forest    0.6825   0.675467  0.681447  0.677530


Optimize Logistic Regression

In [21]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'l1_ratio': [0, 1],
    'solver': ['saga']
}

grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
)

grid_search.fit(X_train_ready, y_train)
best_lr = grid_search.best_estimator_

Evaluate tuned model

In [25]:
y_pred = best_lr.predict(X_test_ready)
y_proba = best_lr.predict_proba(X_test_ready)

print('Best Params:', grid_search.best_params_)
print(f'CV F1-Score:   {grid_search.best_score_:.4f}')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Test F1-Score: {f1_score(y_test, y_pred, average="macro"):.4f}')

Best Params: {'C': 1, 'l1_ratio': 1, 'solver': 'saga'}
CV F1-Score:   0.7584
Test Accuracy: 0.7675
Test F1-Score: 0.7671


Saving the model

In [26]:
best_model_path = models_dir / 'best_model.joblib'
joblib.dump(best_lr, best_model_path)
print(f'Saved best model to: {best_model_path}')

Saved best model to: C:\Users\lioub\PycharmProjects\MindPulse-AI\models\best_model.joblib
